# Using the JWST Exoplanet List (JWEL)
--------
**Author**: Nestor Espinoza (nespinoza@stsci.edu)

## 1. Loading and exploring the data

Let's first read in the dictionary:

In [1]:
import numpy as np
from utils import read_file

In [2]:
thedict = read_file('documents/all.csv')

Let's check the dictionary keys:

In [3]:
thedict.keys()

dict_keys(['Cycle', 'PID', 'Star', 'J-magnitude', 'Planet', 'Stellar Radius (Solar Radii)', 'Stellar Mass (Solar Mass)', 'Stellar Teff (K)', 'Distance (pc)', 'Planet Mass (Earth masses)', 'Planet Radius (Earth radii)', 'Planet Teq (K)', 'Planet Period (days)', 'Planet semi-major axis (AU)', 'Observation', 'Instrument/Mode', 'Filter', 'Science Mode', 'Sub-science theme', 'Seconds on target', 'Target multiplier'])

All right. Let's do some cuts. First, let's get the list of names of all the exoplanets observed by JWST:

In [4]:
jwst_exoplanets = set(thedict['Planet'])

In [5]:
len(jwst_exoplanets)

195

These are the number of unique exoplanets. 

## 2. How many sub-Neptunes in JWST?

Let's check which of those are sub-neptunes --- i.e., with radii between 1.7 and 4. To this end, first convert all the radii from string to float. First, identify empty values and set them to 9999: 

In [6]:
idx = np.where(thedict['Planet Radius (Earth radii)'] == '')[0]
thedict['Planet Radius (Earth radii)'][idx] = 9999.

And now convert:

In [7]:
thedict['Planet Radius (Earth radii)'] = thedict['Planet Radius (Earth radii)'].astype('float')

And find which one have radii between 1.7 and 4:

In [8]:
idx = np.where( (thedict['Planet Radius (Earth radii)'] > 1.7) & (thedict['Planet Radius (Earth radii)'] < 4) )[0]

In [9]:
len(idx)

101

All right --- but those are not unique. Uniquify-them:

In [10]:
jwst_subneptunes = set(thedict['Planet'][idx])

In [11]:
len(jwst_subneptunes)

35

Let's get the list of those sub-Neptunes, and their associated program IDs:

In [12]:
for sn in jwst_subneptunes:

    first_time = True
    for i in range(len(thedict['Planet'])):

        if thedict['Planet'][i] == sn:

            if first_time:

                # Extract planet properties:
                pm, pr, pt, st, sr = float(thedict['Planet Mass (Earth masses)'][i]),\
                                     float(thedict['Planet Radius (Earth radii)'][i]),\
                                     float(thedict['Planet Teq (K)'][i]),\
                                     float(thedict['Stellar Teff (K)'][i]),\
                                     float(thedict['Stellar Radius (Solar Radii)'][i])
                print(sn, 
                      '(mass {0:.2f} ME; radius {1:.2f} RE; Teq {2:.0f} K; stellar Teff {3:.0f} K; stellar radius {4:.2f} Rsun) is being observed...'.format(pm, pr, pt, st, sr))

                first_time = False
            
            print('...in cycle ', 
                  int(thedict['Cycle'][i]),
                  ' by PID', int(thedict['PID'][i]), 
                  'with',thedict['Instrument/Mode'][i])

    print('\n')

TOI-776 b (mass 5.00 ME; radius 1.80 RE; Teq 520 K; stellar Teff 3725 K; stellar radius 0.55 Rsun) is being observed...
...in cycle  1  by PID 2512 with NIRSpec/G395H
...in cycle  1  by PID 2512 with NIRSpec/G395H


HD 20329 b (mass 7.42 ME; radius 1.72 RE; Teq 2141 K; stellar Teff 5596 K; stellar radius 1.13 Rsun) is being observed...
...in cycle  3  by PID 4818 with MIRI/LRS


GJ 3090 b (mass 3.34 ME; radius 2.13 RE; Teq 693 K; stellar Teff 3556 K; stellar radius 0.52 Rsun) is being observed...
...in cycle  2  by PID 4098 with NIRISS/SOSS
...in cycle  2  by PID 4098 with NIRISS/SOSS
...in cycle  2  by PID 4098 with NIRSpec/G395H
...in cycle  2  by PID 4098 with NIRSpec/G395H


TOI-2076 b (mass 16.10 ME; radius 2.77 RE; Teq 688 K; stellar Teff 5192 K; stellar radius 0.80 Rsun) is being observed...
...in cycle  3  by PID 5959 with NIRSpec/G395H
...in cycle  3  by PID 5959 with NIRISS/SOSS


WASP-47 e (mass 9.00 ME; radius 1.83 RE; Teq 2200 K; stellar Teff 5371 K; stellar radius 1.16 Rs

## 3. How much time spent on-target for TRAPPIST-1?

In [13]:
idx = np.where(thedict['Star']=='TRAPPIST-1')[0]

In [14]:
np.sum(thedict['Seconds on target'][idx] * thedict['Target multiplier'][idx]) / 3600.

260.774365

That's time-on-target. Overheads for TRAPPIST-1 are sizeable, however, and we can estimate them. For instance, for L. Kreidberg's program (PID 2304) the time-on-target was:

In [18]:
idx = np.where(thedict['PID']==2304)[0]

In [19]:
np.sum(thedict['Seconds on target'][idx] * thedict['Target multiplier'][idx]) / 3600.

12.839183333333333

But the actual charged time (https://www.stsci.edu/jwst/science-execution/approved-programs/general-observers/cycle-1-go) was 17.9. That's a fraction of...

In [21]:
17.9/12.839183333333333

1.3941696707085471

About 40% of the time spent on-sky. Let's calculate the same for the program of Allen & Espinoza (who use NIRSpec):

In [22]:
idx = np.where(thedict['PID']==6456)[0]

In [23]:
np.sum(thedict['Seconds on target'][idx] * thedict['Target multiplier'][idx]) / 3600.

39.75062277777778

Here, charged time (https://www.stsci.edu/jwst/science-execution/approved-programs/general-observers/cycle-3-go) is 52.1. Again:

In [24]:
52.1/39.75062277777778

1.310671289133262

Of order 30% in overheads. Let's use this as a lower limit --- the total (charged) time spent on TRAPPIST-1 is, thus:

In [25]:
260.774365 * 1.3

339.0066745

And the upper limit:

In [27]:
260.774365 * 1.4

365.08411099999995

So time charged to observe TRAPPIST-1 is in the order of 350 hours in total.

### 3.1 Which programs are observing which TRAPPIST-1 planets?

Let's see:

In [44]:
jwst_t1 = set(thedict['Planet'][idx])
print(jwst_t1)

{'TRAPPIST-1 c', 'TRAPPIST-1 h', 'TRAPPIST-1 g', 'TRAPPIST-1 f', 'TRAPPIST-1 b', 'TRAPPIST-1 e', 'TRAPPIST-1 d'}


So, all TRAPPIST-1 planets are being observed. By which programs?

In [46]:
jwst_t1 = list(jwst_t1)
jwst_t1.sort()

for t1p in jwst_t1:

    first_time = True
    for i in range(len(thedict['Planet'])):

        if thedict['Planet'][i] == t1p:

            if first_time:

                # Extract planet properties:
                pm, pr, pt, st, sr = float(thedict['Planet Mass (Earth masses)'][i]),\
                                     float(thedict['Planet Radius (Earth radii)'][i]),\
                                     float(thedict['Planet Teq (K)'][i]),\
                                     float(thedict['Stellar Teff (K)'][i]),\
                                     float(thedict['Stellar Radius (Solar Radii)'][i])
                print(t1p, 
                      '(mass {0:.2f} ME; radius {1:.2f} RE; Teq {2:.0f} K; stellar Teff {3:.0f} K; stellar radius {4:.2f} Rsun) is being observed...'.format(pm, pr, pt, st, sr))

                first_time = False
            
            print('...in cycle ', 
                  int(thedict['Cycle'][i]),
                  ' by PID', int(thedict['PID'][i]), 
                  'with',thedict['Instrument/Mode'][i])

    print('\n')

TRAPPIST-1 b (mass 1.37 ME; radius 1.12 RE; Teq 398 K; stellar Teff 2566 K; stellar radius 0.12 Rsun) is being observed...
...in cycle  1  by PID 1981 with NIRSpec/PRISM
...in cycle  1  by PID 2420 with NIRSpec/PRISM
...in cycle  1  by PID 2589 with NIRISS/SOSS
...in cycle  1  by PID 2589 with NIRISS/SOSS
...in cycle  2  by PID 3077 with MIRI/Imaging
...in cycle  2  by PID 3077 with MIRI/Imaging
...in cycle  3  by PID 5191 with MIRI/Imaging
...in cycle  3  by PID 5191 with MIRI/Imaging
...in cycle  3  by PID 5191 with MIRI/Imaging
...in cycle  3  by PID 5191 with MIRI/Imaging
...in cycle  3  by PID 6456 with NIRSpec/PRISM
...in cycle  3  by PID 6456 with NIRSpec/PRISM
...in cycle  3  by PID 6456 with NIRSpec/PRISM
...in cycle  3  by PID 6456 with NIRSpec/PRISM
...in cycle  3  by PID 6456 with NIRSpec/PRISM
...in cycle  3  by PID 6456 with NIRSpec/PRISM
...in cycle  1  by PID 1177 with MIRI/Imaging
...in cycle  1  by PID 1177 with MIRI/Imaging
...in cycle  1  by PID 1177 with MIRI/Imagi

## 4. Rocky exoplanets and JWST

Let's now switch our attention to rocky planets. First, cut by radius and figure out the programs observing those objects:

In [51]:
idx = np.where( (thedict['Planet Radius (Earth radii)'] < 2) )[0]

In [52]:
jwst_rocky = set(thedict['Planet'][idx])

In [53]:
len(jwst_rocky)

54

Let's check the targets and their programs:

In [54]:
for rp in jwst_rocky:

    first_time = True
    for i in range(len(thedict['Planet'])):

        if thedict['Planet'][i] == rp:

            if first_time:

                # Extract planet properties:
                pm, pr, pt, st, sr = float(thedict['Planet Mass (Earth masses)'][i]),\
                                     float(thedict['Planet Radius (Earth radii)'][i]),\
                                     float(thedict['Planet Teq (K)'][i]),\
                                     float(thedict['Stellar Teff (K)'][i]),\
                                     float(thedict['Stellar Radius (Solar Radii)'][i])
                print(rp, 
                      '(mass {0:.2f} ME; radius {1:.2f} RE; Teq {2:.0f} K; stellar Teff {3:.0f} K; stellar radius {4:.2f} Rsun) is being observed...'.format(pm, pr, pt, st, sr))

                first_time = False
            
            print('...in cycle ', 
                  int(thedict['Cycle'][i]),
                  ' by PID', int(thedict['PID'][i]), 
                  'with',thedict['Instrument/Mode'][i])

    print('\n')

LHS 3844 b (mass 2.29 ME; radius 1.30 RE; Teq 805 K; stellar Teff 3036 K; stellar radius 0.19 Rsun) is being observed...
...in cycle  1  by PID 1846 with MIRI/LRS
...in cycle  1  by PID 1846 with MIRI/LRS
...in cycle  1  by PID 1846 with MIRI/LRS
...in cycle  2  by PID 4008 with NIRSpec/G395H


TRAPPIST-1 g (mass 1.32 ME; radius 1.13 RE; Teq 197 K; stellar Teff 2566 K; stellar radius 0.12 Rsun) is being observed...
...in cycle  1  by PID 2589 with NIRSpec/PRISM
...in cycle  1  by PID 2589 with NIRSpec/PRISM


TOI-776 b (mass 5.00 ME; radius 1.80 RE; Teq 520 K; stellar Teff 3725 K; stellar radius 0.55 Rsun) is being observed...
...in cycle  1  by PID 2512 with NIRSpec/G395H
...in cycle  1  by PID 2512 with NIRSpec/G395H


L98-59 b (mass 0.40 ME; radius 0.85 RE; Teq 627 K; stellar Teff 3415 K; stellar radius 0.30 Rsun) is being observed...
...in cycle  2  by PID 3942 with NIRSpec/G395H
...in cycle  2  by PID 3942 with NIRSpec/G395H
...in cycle  2  by PID 3942 with NIRSpec/G395H
...in cyc